In [ ]:
pip install -U "monai[all]"

In [ ]:
# %% [markdown]
# # UCSF-PDGM Metadata Inspection (Corrected Columns)
# Explore and preprocess clinical metadata for tumor classification and survival prediction.

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# %%
# === Load Data ===
path = r"C:\Users\Kouts\Downloads\UCSF-PDGM-metadata_v5 (1).csv" 
df = pd.read_csv(path)
print("Shape:", df.shape)
df.head()

# %%
# === Ensure Correct Column Names ===
expected_cols = [
    'ID', 'Sex', 'Age at MRI', 'WHO CNS Grade',
    'Final pathologic diagnosis (WHO 2021)', 'MGMT status', 'MGMT index',
    '1p/19q', 'IDH', '1-dead 0-alive', 'OS', 'EOR',
    'Biopsy prior to imaging', 'BraTS21 ID',
    'BraTS21 Segmentation Cohort', 'BraTS21 MGMT Cohort'
]

missing = [c for c in expected_cols if c not in df.columns]
if missing:
    print("⚠️ Missing columns:", missing)
else:
    print("✅ All expected columns found.")

# %%
# === Rename Columns (Python-friendly) ===
rename_map = {
    'Age at MRI': 'Age',
    'WHO CNS Grade': 'Grade',
    'Final pathologic diagnosis (WHO 2021)': 'Diagnosis',
    'MGMT status': 'MGMT_Status',
    'MGMT index': 'MGMT_Index',
    '1p/19q': 'Codeletion_1p19q',
    '1-dead 0-alive': 'Survival_Status',
    'Biopsy prior to imaging': 'PreImaging_Biopsy',
    'BraTS21 ID': 'BraTS21_ID',
    'BraTS21 Segmentation Cohort': 'Segmentation_Cohort',
    'BraTS21 MGMT Cohort': 'MGMT_Cohort'
}

df.rename(columns=rename_map, inplace=True)
df.head(2)

# %%
# === Basic Info ===
df.info()
df.describe(include='all').T

# %%
# === Missing Values Overview ===
plt.figure(figsize=(10,5))
sns.heatmap(df.isna(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()

df.isna().sum().sort_values(ascending=False)

# %%
# === Data Types Summary ===
numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

print("Numeric Columns:", list(numeric_cols))
print("Categorical Columns:", list(categorical_cols))

# %%
# === Distribution: Tumor Grade ===
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Grade', palette='viridis')
plt.title("Distribution of WHO CNS Grades")
plt.show()

# %%
# === Distribution: Diagnosis ===
plt.figure(figsize=(10,5))
df['Diagnosis'].value_counts().plot(kind='bar')
plt.title("Final Pathologic Diagnosis (WHO 2021)")
plt.ylabel("Count")
plt.show()

# %%
# === Distribution: MGMT Status ===
plt.figure(figsize=(5,4))
sns.countplot(data=df, x='MGMT_Status', palette='magma')
plt.title("MGMT Promoter Methylation Status")
plt.show()

# %%
# === Distribution: IDH & 1p/19q ===
fig, axes = plt.subplots(1, 2, figsize=(10,4))
sns.countplot(data=df, x='IDH', ax=axes[0], palette='cool')
axes[0].set_title("IDH Mutation Status")

sns.countplot(data=df, x='Codeletion_1p19q', ax=axes[1], palette='coolwarm')
axes[1].set_title("1p/19q Codeletion Status")
plt.show()

# %%
# === Survival Overview (if available) ===
if 'OS' in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df['OS'].dropna(), bins=20, kde=True)
    plt.title("Overall Survival (days)")
    plt.xlabel("OS (Days)")
    plt.show()

# %%
# === Classification Targets ===
print("Unique Grades:", df['Grade'].unique())
print("Unique Diagnoses:", df['Diagnosis'].unique())

# %%
# === Label Encoding for ML Tasks ===
df_clean = df.copy()

df_clean['Grade_Label'] = df_clean['Grade'].astype('category').cat.codes
df_clean['Diagnosis_Label'] = df_clean['Diagnosis'].astype('category').cat.codes
df_clean['MGMT_Label'] = df_clean['MGMT_Status'].astype('category').cat.codes

print(df_clean[['ID', 'Grade', 'Grade_Label', 'Diagnosis', 'Diagnosis_Label', 'MGMT_Status', 'MGMT_Label']].head())

# %%
# === Save Cleaned Data ===
df_clean.to_csv("UCSF_PDGM_metadata_clean.csv", index=False)
print("✅ Saved cleaned metadata as UCSF_PDGM_metadata_clean.csv")


In [ ]:
from sklearn.model_selection import train_test_split

# Ensure the grade label exists
if 'Grade_Label' not in df_clean.columns:
    df_clean['Grade_Label'] = df_clean['Grade'].astype('category').cat.codes

# Drop missing grades
df_strat = df_clean.dropna(subset=['Grade'])
print(f"Remaining cases after dropping missing grades: {len(df_strat)}")

# First: train+val vs test (80/20)
trainval_df, test_df = train_test_split(
    df_strat,
    test_size=0.2,
    stratify=df_strat['Grade_Label'],
    random_state=42
)

# Then: split train+val into train (75%) and val (25%)
train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.25,  # 25% of 80% = 20% total
    stratify=trainval_df['Grade_Label'],
    random_state=42
)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# %%
# === Check Distribution Consistency ===
def show_split_distribution(df, label_col='Grade'):
    return df[label_col].value_counts(normalize=True).sort_index()

print("\nGrade Distribution:")
print("Train:\n", show_split_distribution(train_df))
print("Val:\n", show_split_distribution(val_df))
print("Test:\n", show_split_distribution(test_df))

# %%
# === Save Splits ===

train_df.to_csv("train_metadata.csv", index=False)
val_df.to_csv("val_metadata.csv", index=False)
test_df.to_csv("test_metadata.csv", index=False)

print("✅ Saved stratified splits to 'splits/' folder.")

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=train_df, x='Grade', palette='viridis')
plt.title("Distribution of WHO CNS Grades")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=val_df, x='Grade', palette='viridis')
plt.title("Distribution of WHO CNS Grades")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=test_df, x='Grade', palette='viridis')
plt.title("Distribution of WHO CNS Grades")
plt.show()

In [ ]:
# %% [markdown]
# # Step 1: Balance Dataset by Grade (Keep 2 & 3, Downsample 4)
# ---
# We keep all Grade 2 and 3 patients and downsample Grade 4 to match Grade 3 count.
# Then we save both the final dataset and the removed patients for reproducibility.

# %%
import os
import pandas as pd
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns


# Define the path to your CSV file
csv_file_path = "UCSF_PDGM_metadata_clean.csv"

# Read the CSV into the DataFrame df_clean
df_clean = pd.read_csv(csv_file_path)
# make sure df_clean is loaded with the columns you mentioned earlier
df = df_clean.dropna(subset=['Grade']).copy()

print("Original class distribution:\n", df['Grade'].value_counts(), "\n")

# find the number of Grade 3 patients
grade3_count = df[df['Grade'] == 3].shape[0]

# separate by class
df_grade2 = df[df['Grade'] == 2]
df_grade3 = df[df['Grade'] == 3]
df_grade4 = df[df['Grade'] == 4]

# downsample Grade 4 to match Grade 3 count
df_grade4_down = resample(df_grade4, replace=False, n_samples=grade3_count +20, random_state=42)

# create balanced dataset
df_balanced = pd.concat([df_grade2, df_grade3, df_grade4_down]).sample(frac=1, random_state=42).reset_index(drop=True)

# identify removed Grade 4 patients
removed_grade4 = df_grade4.loc[~df_grade4['ID'].isin(df_grade4_down['ID'])]

# save datasets
os.makedirs("splits", exist_ok=True)
df_balanced.to_csv("splits/balanced_metadata_gradeaware.csv", index=False)
removed_grade4.to_csv("splits/removed_grade4_patients.csv", index=False)

print("✅ Balanced dataset created.")
print(df_balanced['Grade'].value_counts())
print(f"💾 Saved balanced dataset and removed IDs to 'splits/' folder.")

# %% [markdown]
# # Step 2: Exploratory Data Analysis (EDA)
# ---
# We explore demographics, survival, molecular markers, and grade distributions.

# %%
# Basic summary
print("=== Summary Statistics ===")
display(df_balanced.describe(include='all').T)

# %%
# Plot class distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df_balanced, x='Grade', palette='viridis')
plt.title("Tumor Grade Distribution (Balanced)")
plt.show()

# %%
# Age distribution by grade
plt.figure(figsize=(6,4))
sns.boxplot(data=df_balanced, x='Grade', y='Age', palette='magma')
plt.title("Age Distribution per Tumor Grade")
plt.show()

# %%
# Sex distribution
plt.figure(figsize=(5,4))
sns.countplot(data=df_balanced, x='Sex', hue='Grade', palette='Set2')
plt.title("Sex Distribution by Tumor Grade")
plt.show()

# %%
# MGMT and IDH mutation status
fig, axes = plt.subplots(1, 2, figsize=(10,4))

sns.countplot(data=df_balanced, x='MGMT_Status', hue='Grade', ax=axes[0], palette='cool')
axes[0].set_title("MGMT Status by Grade")

sns.countplot(data=df_balanced, x='IDH', hue='Grade', ax=axes[1], palette='rocket')
axes[1].set_title("IDH Status by Grade")

plt.tight_layout()
plt.show()

# %%
# Survival analysis overview
plt.figure(figsize=(6,4))
sns.boxplot(data=df_balanced, x='Grade', y='OS', palette='Spectral')
plt.title("Overall Survival by Tumor Grade")
plt.ylabel("Overall Survival (days)")
plt.show()

# %%
# Diagnosis distribution
plt.figure(figsize=(10, 5)) # Adjust figsize if diagnosis names are long
sns.countplot(data=df_balanced, y='Diagnosis', hue='Grade', palette='tab10', order = df_balanced['Diagnosis'].value_counts().index) # Plot horizontally if names are long
plt.title("Diagnosis Distribution by Tumor Grade")
plt.xlabel("Count")
plt.ylabel("Diagnosis")
plt.tight_layout() # Adjust layout
plt.show()
# %%
# Correlation heatmap for numeric columns
num_cols = df_balanced.select_dtypes(include=['number'])
plt.figure(figsize=(8,6))
sns.heatmap(num_cols.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlations")
plt.show()

# Diagnosis counts
print("\n=== Diagnosis Distribution (Counts) ===")
diagnosis_counts = df_balanced['Diagnosis'].value_counts()
print(diagnosis_counts)

# Optional: Plotting just the counts without hue
plt.figure(figsize=(10, 5)) # Adjust figsize if diagnosis names are long
sns.countplot(data=df_balanced, y='Diagnosis', palette='tab10', order = diagnosis_counts.index) # Plot horizontally if names are long
plt.title("Diagnosis Distribution (Balanced Dataset)")
plt.xlabel("Count")
plt.ylabel("Diagnosis")
plt.tight_layout() # Adjust layout
plt.show()
# %%
print("✅ EDA completed.")



In [ ]:
df_balanced.head()

In [ ]:
import pandas as pd

df_balanced = pd.read_csv("splits/balanced_metadata_gradeaware.csv")
df_balanced.info()

In [ ]:
# --- Imports ---
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

# === Paths (change these to your local files) ===
volume_path = r"C:\Users\Kouts\Downloads\Volumes\UCSF-PDGM-0004_T2.nii.gz"
mask_path   = r"C:\Users\Kouts\Downloads\Segmentations\UCSF-PDGM-0004_tumor_segmentation.nii.gz"

# --- Load NIfTI files ---
vol_img = nib.load(volume_path)
mask_img = nib.load(mask_path)

vol_data = vol_img.get_fdata()
mask_data = mask_img.get_fdata()

print("Volume shape:", vol_data.shape)
print("Segmentation shape:", mask_data.shape)
assert vol_data.shape == mask_data.shape, "❌ Volume and mask shapes do not match!"

# --- Pick central slice indices ---
mid_slices = [s//2 for s in vol_data.shape]

# --- Prepare three orthogonal planes ---
axial_vol = vol_data[:, :, mid_slices[2]]
coronal_vol = vol_data[:, mid_slices[1], :]
sagittal_vol = vol_data[mid_slices[0], :, :]

axial_mask = mask_data[:, :, mid_slices[2]]
coronal_mask = mask_data[:, mid_slices[1], :]
sagittal_mask = mask_data[mid_slices[0], :, :]

# --- Visualization ---
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

titles = ['Axial (Top→Bottom)', 'Coronal (Front→Back)', 'Sagittal (Side→Side)']
images = [(axial_vol, axial_mask), (coronal_vol, coronal_mask), (sagittal_vol, sagittal_mask)]

for ax, (img, msk), title in zip(axs, images, titles):
    ax.imshow(np.rot90(img), cmap='gray')
    ax.imshow(np.rot90(msk), cmap='autumn', alpha=0.4)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- Imports ---
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# === Paths (change these to your local files) ===
volume_path = r"C:\Users\Kouts\Downloads\Volumes\UCSF-PDGM-0004_T2.nii.gz"
mask_path   = r"C:\Users\Kouts\Downloads\Segmentations\UCSF-PDGM-0004_tumor_segmentation.nii.gz"


# --- Load volume and mask ---
vol_img = nib.load(volume_path)
mask_img = nib.load(mask_path)

vol = vol_img.get_fdata()
mask = mask_img.get_fdata()

print("Volume shape:", vol.shape)
print("Segmentation shape:", mask.shape)
assert vol.shape == mask.shape, "❌ Shapes do not match!"

# --- Visualization function ---
def show_slices(axial_idx, coronal_idx, sagittal_idx):
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    titles = ['Axial', 'Coronal', 'Sagittal']
    slices = [
        (vol[:, :, axial_idx], mask[:, :, axial_idx]),
        (vol[:, coronal_idx, :], mask[:, coronal_idx, :]),
        (vol[sagittal_idx, :, :], mask[sagittal_idx, :, :])
    ]
    
    for ax, (img, msk), title in zip(axs, slices, titles):
        ax.imshow(np.rot90(img), cmap='gray')
        ax.imshow(np.rot90(msk), cmap='autumn', alpha=0.4)
        ax.set_title(f"{title} view")
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# --- Sliders for each dimension ---
interact(
    show_slices,
    axial_idx=IntSlider(min=0, max=vol.shape[2]-1, step=1, value=vol.shape[2]//2),
    coronal_idx=IntSlider(min=0, max=vol.shape[1]-1, step=1, value=vol.shape[1]//2),
    sagittal_idx=IntSlider(min=0, max=vol.shape[0]-1, step=1, value=vol.shape[0]//2)
);


In [ ]:
np.unique(mask)

In [ ]:
import os
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
from monai.transforms import (
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Orientationd,
    ScaleIntensityRanged,
    CropForegroundd,
    RandSpatialCropd,
    Compose
)

# === example file paths ===
volume_path = r"C:\Users\Kouts\Downloads\Volumes\UCSF-PDGM-0004_T2.nii.gz"
mask_path   = r"C:\Users\Kouts\Downloads\Segmentations\UCSF-PDGM-0004_tumor_segmentation.nii.gz"

# === define preprocessing + ROI crop ===
roi_size = (192, 192, 192)

transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),  # remove background
    RandSpatialCropd(keys=["image", "label"], roi_size=roi_size, random_center=True, random_size=False)
])

data_dict = {"image": volume_path, "label": mask_path}
sample = transforms(data_dict)

print("Transformed image shape:", sample["image"].shape)
print(data_dict)
img = sample["image"][0]  # shape (H, W, D)
mask = sample["label"][0]

print("Cropped image shape:", img.shape)

# === visualize 3 orthogonal slices ===
mid_slices = [s//2 for s in img.shape]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img[mid_slices[0], :, :], cmap="gray")
axes[0].imshow(mask[mid_slices[0], :, :], alpha=0.3)
axes[0].set_title("Sagittal")

axes[1].imshow(img[:, mid_slices[1], :], cmap="gray")
axes[1].imshow(mask[:, mid_slices[1], :], alpha=0.3)
axes[1].set_title("Coronal")

axes[2].imshow(img[:, :, mid_slices[2]], cmap="gray")
axes[2].imshow(mask[:, :, mid_slices[2]], alpha=0.3)
axes[2].set_title("Axial")

plt.tight_layout()
plt.show()


In [ ]:
import os
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
from monai.transforms import (
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Orientationd,
    ScaleIntensityRanged,
    CropForegroundd,
    RandSpatialCropd,
    Compose,
    RandSpatialCropSamplesd,
    ToTensord
)

# === example file paths ===
volume_path = r"C:\Users\Kouts\Downloads\Volumes\UCSF-PDGM-0004_T2.nii.gz"
mask_path   = r"C:\Users\Kouts\Downloads\Segmentations\UCSF-PDGM-0004_tumor_segmentation.nii.gz"

sample = {"image": volume_path, "label": mask_path}
# === define preprocessing + ROI crop ===
roi_size = (128, 128, 64)
num_samples = 15  # visualize 3 crops

transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=2000, b_min=0, b_max=1, clip=True),
    RandSpatialCropSamplesd(
        keys=["image", "label"],
        roi_size=roi_size,
        num_samples=num_samples,
        random_center=True,
        random_size=False
    ),
    CropForegroundd(keys=["image", "label"], source_key="image"),  # remove background
    RandSpatialCropd(keys=["image", "label"], roi_size=roi_size, random_center=True, random_size=False),
    ToTensord(keys=["image", "label"])
])

samples = transforms(sample)

def show_slice(image, mask, title):
    """Show the middle slice of a 3D volume and its mask overlay"""
    image_np = image[0].cpu().numpy()
    mask_np = mask[0].cpu().numpy()
    mid = image_np.shape[2] // 2 # middle slice along z-axis

    plt.figure(figsize=(5,5))
    plt.imshow(image_np[:, :, mid], cmap="gray")
    plt.imshow(mask_np[:, :, mid], alpha=0.4, cmap="Reds")
    plt.title(title)
    plt.axis("off")
    plt.show()

for i, s in enumerate(samples):
    show_slice(s["image"], s["label"], f"Random crop #{i+1}")

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from monai.data import Dataset, CacheDataset, DataLoader
from monai.transforms import (
    LoadImaged, EnsureChannelFirstd, NormalizeIntensityd, RandSpatialCropd,
    RandFlipd, RandRotate90d, Compose, ScaleIntensityRanged
)
from monai.networks.nets import SwinUNETR
from monai.losses import SSIMLoss
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
data_dir = r"C:\Users\Kouts\Downloads\Volumes"  # or your GDrive path
volume_files = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".nii") or f.endswith(".nii.gz")])

data_dicts = [{"image": f} for f in volume_files]
print(f"Found {len(data_dicts)} volumes.")
train_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityRanged(keys=["image"], a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
    RandSpatialCropd(keys=["image"], roi_size=(96, 96, 96), random_size=False),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
    RandRotate90d(keys=["image"], prob=0.5, spatial_axes=(0, 1)),
])

train_ds = CacheDataset(data=data_dicts, transform=train_transforms, cache_rate=0.1)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
print(f"Train dataset size: {len(train_ds)}")

In [ ]:
for batch in train_loader:
    img = batch["image"]
    print("Batch shape:", img.shape)  # (B, C, H, W, D)
    break

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
from monai.networks.nets import SwinUNETR

encoder = SwinUNETR(

    in_channels=1,
    out_channels=1,
    feature_size=48,
    use_checkpoint=True,  # optional: saves memory
).to(device)

In [ ]:
from monai.networks.nets import SwinUNETR
import torch.nn as nn

class DualPretrainModel(nn.Module):
    def __init__(self, img_size=(96, 96, 96), feature_size=48):
        super().__init__()
        self.encoder = SwinUNETR(
            
            in_channels=1,
            out_channels=feature_size,  # latent representation
            feature_size=feature_size,
            use_checkpoint=True,
        )
        self.mae_decoder = nn.Conv3d(feature_size, 1, kernel_size=3, padding=1)
        self.projection_head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(feature_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
        )

    def forward(self, x, mask=None):
        features = self.encoder(x)
        recon = self.mae_decoder(features)
        proj = self.projection_head(features)
        return recon, proj

In [ ]:
import torch.nn.functional as F

def contrastive_loss(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    logits = torch.mm(z1, z2.t()) / temperature
    labels = torch.arange(z1.size(0)).to(z1.device)
    return F.cross_entropy(logits, labels)

def pretrain_step(model, imgs1, imgs2, optimizer):
    recon1, proj1 = model(imgs1)
    recon2, proj2 = model(imgs2)

    mae_loss = F.mse_loss(recon1, imgs1) + F.mse_loss(recon2, imgs2)
    cont_loss = contrastive_loss(proj1, proj2)
    loss = mae_loss + 0.1 * cont_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [ ]:
model = DualPretrainModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(1):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        imgs = batch["image"]
        # Augmentations for contrastive pairs
        imgs1 = imgs + 0.05 * torch.randn_like(imgs)
        imgs2 = torch.flip(imgs, dims=[4])  # simple spatial flip

        loss = pretrain_step(model, imgs1, imgs2, optimizer)
        running_loss += loss
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator
import os
from glob import glob
import numpy as np
import pandas as pd # Ensure pandas is imported

# --- Configuration ---
# *** Verify this path points to your training history CSV ***

history_csv_path = r"C:\Users\Kouts\Downloads\training_history.csv"  # Change to your actual CSV path
# --- Manually Extracted Data from Logs (Epochs 1-12) ---
# Assuming Validation ran EVERY epoch (VAL_INTERVAL = 1) based on logs
history_data = {
    "epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "train_loss": [
        1.1278, 0.9290, 0.8448, 0.7813, 0.7368, 0.6893,
        0.6445, 0.6134, 0.6050, 0.5952, 0.5883, 0.5871
    ],
    # Validation loss was not printed in the logs, represented as None/NaN
    "val_loss": [
        None, None, None, None, None, None,
        None, None, None, None, None, None
        # Replace None with actual values if you find them logged elsewhere
    ],
    # Validation Dice score logged every epoch
    "val_dice": [
        0.3291, 0.6399, 0.7153, 0.6953, 0.6740, 0.7350,
        0.7006, 0.7078, 0.7099, 0.7069, 0.7392, 0.7392
    ]
}

# Function to read history from CSV
def load_history_from_csv(csv_path):
    """Loads training history from a CSV file."""
    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_dice": []
    }
    try:
        if os.path.exists(csv_path):
            df_history = pd.read_csv(csv_path)
            history = df_history.to_dict('list')
            print(f"Successfully loaded history from: {csv_path}")
            # Ensure all keys exist even if CSV was partial (though unlikely now)
            for key in ["epoch", "train_loss", "val_loss", "val_dice"]:
                if key not in history: history[key] = [np.nan] * len(history["epoch"])
        else:
            print(f"Warning: History file not found at {csv_path}. Plot will be empty.")
    except Exception as e:
        print(f"Error reading history CSV: {e}")
    return history

# Load the data
history = load_history_from_csv(history_csv_path)

# --- Plotting ---
plt.style.use('seaborn-v0_8-darkgrid')
plt.figure("Training & Validation Curves", figsize=(12, 5))

# --- Subplot 1: Epoch Average Training Loss ---
ax_train = plt.subplot(1, 2, 1)
plt.title("Epoch Average Training Loss")
if history.get('train_loss') and history.get('epoch') and not all(np.isnan(history['train_loss'])):
    train_epochs = history['epoch']
    train_losses = history['train_loss']
    
    ax_train.plot(train_epochs, train_losses, marker='o', linestyle='-', color="red", label='Avg Train Loss')
    ax_train.set_xlabel("Epoch")
    ax_train.set_ylabel("DiceCE Loss")
    ax_train.grid(True, linestyle=':')
    ax_train.legend()
    ax_train.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
else:
    ax_train.text(0.5, 0.5, 'Train Loss Data Not Found', ha='center', va='center', fontsize=12)
    ax_train.set_xlabel("Epoch")
    ax_train.set_ylabel("DiceCE Loss")

# --- Subplot 2: Validation Mean Dice ---
ax_val_dice = plt.subplot(1, 2, 2)
plt.title("Average Validation Dice per Epoch")
if history.get('val_dice') and history.get('epoch') and not all(np.isnan(history['val_dice'])):
    # Filter out NaNs for plotting lines (epochs where validation didn't run)
    val_epochs_dice = [e for e, vd in zip(history['epoch'], history['val_dice']) if not np.isnan(vd)]
    val_dices_data = [vd for vd in history['val_dice'] if not np.isnan(vd)]
    
    if val_epochs_dice: # Only plot if there is valid dice data
        ax_val_dice.plot(val_epochs_dice, val_dices_data, marker='s', linestyle='-', color="green", label='Avg Val Dice')
        ax_val_dice.set_xlabel("Epoch")
        ax_val_dice.set_ylabel("Dice Score")
        ax_val_dice.set_ylim(0, max(1.0, max(val_dices_data)*1.1) if val_dices_data else 1.0) # Set y-limit
        ax_val_dice.grid(True, linestyle=':')
        ax_val_dice.legend()
        ax_val_dice.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    else:
        ax_val_dice.text(0.5, 0.5, 'Validation Dice Data Not Found', ha='center', va='center', fontsize=12)
        ax_val_dice.set_xlabel("Epoch")
        ax_val_dice.set_ylabel("Dice Score")
else:
     ax_val_dice.text(0.5, 0.5, 'Validation Dice Data Not Found', ha='center', va='center', fontsize=12)
     ax_val_dice.set_xlabel("Epoch")
     ax_val_dice.set_ylabel("Dice Score")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
# Remove tensorboard imports if not needed
# from tensorboard.backend.event_processing import event_accumulator
import os
from glob import glob
import numpy as np
import pandas as pd # Ensure pandas is imported
from IPython.display import display # For better DataFrame printing

# --- Configuration ---
# *** Path to the CSV file saved by your training loop ***
history_csv_path = r"C:\Users\Kouts\Downloads\training_history.csv" # Use the actual saved file

# --- Manually Extracted Data from Logs (Epochs 1-12) ---
# Validation ran every epoch in these logs
history_data_manual = {
    "epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "train_loss": [
        1.1278, 0.9290, 0.8448, 0.7813, 0.7368, 0.6893,
        0.6445, 0.6134, 0.6050, 0.5952, 0.5883, 0.5871
    ],
    "val_loss": [ # Fill with NaN as these were not logged
        np.nan, np.nan, np.nan, np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
    ],
    "val_dice": [
        0.3291, 0.6399, 0.7153, 0.6953, 0.6740, 0.7350,
        0.7006, 0.7078, 0.7099, 0.7069, 0.7392, 0.7392
    ]
}

# --- Load Data from CSV (Epochs 13 onwards) ---
history_from_csv = { # Initialize empty in case file not found
    "epoch": [], "train_loss": [], "val_loss": [], "val_dice": []
}
try:
    if os.path.exists(history_csv_path):
        df_history_csv = pd.read_csv(history_csv_path)
        # Ensure it only contains data *after* the manual data
        df_history_csv = df_history_csv[df_history_csv['epoch'] > history_data_manual['epoch'][-1]]
        history_from_csv = df_history_csv.to_dict('list')
        print(f"Successfully loaded history from CSV (Epochs > {history_data_manual['epoch'][-1]}): {history_csv_path}")
        # Ensure all keys exist
        for key in ["epoch", "train_loss", "val_loss", "val_dice"]:
            if key not in history_from_csv: history_from_csv[key] = [np.nan] * len(history_from_csv["epoch"])
    else:
        print(f"Warning: History file not found at {history_csv_path}. Plotting only manual data.")
except Exception as e:
    print(f"Error reading history CSV: {e}")

# --- Combine Manual Data and CSV Data ---
combined_history = {}
for key in history_data_manual.keys():
    # Extend the manual list with the list loaded from CSV
    combined_history[key] = history_data_manual[key] + history_from_csv.get(key, []) # Use .get for safety

print(f"\nCombined history covers epochs {combined_history['epoch'][0]} to {combined_history['epoch'][-1]}.")
# Optional: Display combined data as DataFrame
# df_combined = pd.DataFrame(combined_history)
# display(df_combined)

# --- Plotting (Uses the combined_history dictionary) ---
history = combined_history # Use the combined data for plotting

plt.style.use('seaborn-v0_8-darkgrid')
plt.figure("Training & Validation Curves", figsize=(12, 5))

# --- Subplot 1: Epoch Average Training Loss ---
ax_train = plt.subplot(1, 2, 1)
plt.title("Epoch Average Training Loss")
if history.get('train_loss') and history.get('epoch') and history['train_loss']: # Check list not empty
    train_epochs = history['epoch']
    train_losses = history['train_loss']
    
    ax_train.plot(train_epochs, train_losses, linestyle='-', color="red", label='Avg Train Loss')
    ax_train.set_xlabel("Epoch")
    ax_train.set_ylabel("DiceCE Loss")
    ax_train.grid(True, linestyle=':')
    ax_train.legend()
    ax_train.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
else:
    ax_train.text(0.5, 0.5, 'Train Loss Data Not Found', ha='center', va='center', fontsize=12)
    ax_train.set_xlabel("Epoch")
    ax_train.set_ylabel("DiceCE Loss")

# --- Subplot 2: Validation Mean Dice ---
ax_val_dice = plt.subplot(1, 2, 2)
plt.title("Average Validation Dice per Epoch")
if history.get('val_dice') and history.get('epoch') and history['val_dice']: # Check list not empty
    # Filter out potential NaNs if validation didn't run every epoch in the CSV part
    val_epochs_dice = [e for e, vd in zip(history['epoch'], history['val_dice']) if not np.isnan(vd)]
    val_dices_data = [vd for vd in history['val_dice'] if not np.isnan(vd)]
    
    if val_epochs_dice: # Only plot if there is valid dice data
        ax_val_dice.plot(val_epochs_dice, val_dices_data, linestyle='-', color="green", label='Avg Val Dice')
        ax_val_dice.set_xlabel("Epoch")
        ax_val_dice.set_ylabel("Dice Score")
        ax_val_dice.set_ylim(0, max(1.0, max(val_dices_data)*1.1) if val_dices_data else 1.0) # Set y-limit
        ax_val_dice.grid(True, linestyle=':')
        ax_val_dice.legend()
        ax_val_dice.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    else:
        ax_val_dice.text(0.5, 0.5, 'Validation Dice Data Not Found', ha='center', va='center', fontsize=12)
        ax_val_dice.set_xlabel("Epoch")
        ax_val_dice.set_ylabel("Dice Score")
else:
     ax_val_dice.text(0.5, 0.5, 'Validation Dice Data Not Found', ha='center', va='center', fontsize=12)
     ax_val_dice.set_xlabel("Epoch")
     ax_val_dice.set_ylabel("Dice Score")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
from IPython.display import display

# --- 1. Configuration ---
# *** ΕΠΙΒΕΒΑΙΩΣΕ ΟΤΙ ΑΥΤΕΣ ΟΙ ΔΙΑΔΡΟΜΕΣ ΕΙΝΑΙ ΣΩΣΤΕΣ ΓΙΑ ΤΟΝ ΦΑΚΕΛΟ "Downloads" ΣΟΥ ***
download_folder_path = r"C:\Users\Kouts\Downloads" 

# Ορίζουμε τις πλήρεις διαδρομές (paths) για κάθε αρχείο CSV
path_pretrained_csv = os.path.join(download_folder_path, "training_history.csv")
path_swin_scratch_csv = os.path.join(download_folder_path, "history_swin_unetr_scratch.csv")
path_unet_scratch_csv = os.path.join(download_folder_path, "history_unet_scratch.csv")
# --- End Configuration ---


# --- 2. Manual Data for Pretrained Model (Epochs 1-12) ---
history_data_manual_pretrained = {
    "epoch": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "train_loss": [
        1.1278, 0.9290, 0.8448, 0.7813, 0.7368, 0.6893,
        0.6445, 0.6134, 0.6050, 0.5952, 0.5883, 0.5871
    ],
    "val_loss": [ # Γέμισμα με NaN αφού δεν καταγράφηκαν
        np.nan, np.nan, np.nan, np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
    ],
    "val_dice": [
        0.3291, 0.6399, 0.7153, 0.6953, 0.6740, 0.7350,
        0.7006, 0.7078, 0.7099, 0.7069, 0.7392, 0.7392
    ]
}

# --- 3. Function to load CSV ---
def load_history_from_csv(csv_path):
    """Φορτώνει το ιστορικό εκπαίδευσης από ένα αρχείο CSV."""
    history = {
        "epoch": [], "train_loss": [], "val_loss": [], "val_dice": []
    }
    try:
        if os.path.exists(csv_path):
            df_history = pd.read_csv(csv_path)
            history = df_history.to_dict('list')
            print(f"Επιτυχής φόρτωση: {os.path.basename(csv_path)}")
            # Έλεγχος για ύπαρξη κλειδιών
            for key in ["epoch", "train_loss", "val_loss", "val_dice"]:
                if key not in history: 
                    print(f"Προσοχή: Η στήλη '{key}' δεν βρέθηκε στο {csv_path}.")
                    history[key] = [np.nan] * len(history["epoch"])
        else:
            print(f"ΣΦΑΛΜΑ: Το αρχείο ιστορικού δεν βρέθηκε στο: {csv_path}.")
    except Exception as e:
        print(f"Σφάλμα ανάγνωσης CSV {csv_path}: {e}")
    return history

# --- 4. Load Data for All Models ---

# 4.1 Load Pretrained (Combine manual + CSV)
history_pretrained_csv = load_history_from_csv(path_pretrained_csv)
history_pretrained = {}
if history_pretrained_csv['epoch']: # Αν το CSV φορτώθηκε σωστά
    # Φιλτράρισμα CSV για να ξεκινά *μετά* τα χειροκίνητα δεδομένα
    df_csv = pd.DataFrame(history_pretrained_csv)
    df_csv = df_csv[df_csv['epoch'] > history_data_manual_pretrained['epoch'][-1]]
    history_pretrained_csv = df_csv.to_dict('list')
    
    # Συνδυασμός
    for key in history_data_manual_pretrained.keys():
        history_pretrained[key] = history_data_manual_pretrained[key] + history_pretrained_csv.get(key, [])
    print(f"Συνδυασμένο ιστορικό (Pretrained). Σύνολο εποχών: {len(history_pretrained['epoch'])}")
else:
    print("Φόρτωση μόνο των χειροκίνητων δεδομένων 1-12 για το Pretrained μοντέλο.")
    history_pretrained = history_data_manual_pretrained

# 4.2 Load Swin-Scratch
history_swin_scratch = load_history_from_csv(path_swin_scratch_csv)

# 4.3 Load UNet-Scratch
history_unet_scratch = load_history_from_csv(path_unet_scratch_csv)

# --- 5. Combine Histories into a Dictionary for Plotting ---
histories_to_plot = {
    "SwinUNETR (Pretrained)": history_pretrained,
    "SwinUNETR (Scratch)": history_swin_scratch,
    "UNet (Scratch)": history_unet_scratch
}

# Ορισμός χρωμάτων
colors = {
    "SwinUNETR (Pretrained)": "blue",
    "SwinUNETR (Scratch)": "green",
    "UNet (Scratch)": "red"
}

# --- 6. Plotting ---
plt.style.use('seaborn-v0_8-darkgrid')
plt.figure("Training & Validation Curves Comparison", figsize=(14, 6))

# --- Γράφημα 1: Μέση Απώλεια Εκπαίδευσης (Epoch Average Training Loss) ---
ax_train = plt.subplot(1, 2, 1)
plt.title("Epoch Average Training Loss Comparison")

for model_name, history in histories_to_plot.items():
    if history.get('train_loss') and history.get('epoch') and history['train_loss']:
        train_epochs = history['epoch']
        train_losses = history['train_loss']
        plot_epochs = [e for e, l in zip(train_epochs, train_losses) if not np.isnan(l)]
        plot_losses = [l for l in train_losses if not np.isnan(l)]
        if plot_epochs:
            ax_train.plot(plot_epochs, plot_losses, linestyle='-', color=colors.get(model_name, 'black'), label=model_name, alpha=0.9, marker='.') # Προσθήκη marker
    else:
        print(f"Δεν βρέθηκαν δεδομένα training loss για {model_name}")

ax_train.set_xlabel("Epoch")
ax_train.set_ylabel("DiceCE Loss")
ax_train.grid(True, linestyle=':')
ax_train.legend()
ax_train.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# --- Γράφημα 2: Μέσο Dice Επικύρωσης (Validation Mean Dice) ---
ax_val_dice = plt.subplot(1, 2, 2)
plt.title("Average Validation Dice per Epoch Comparison")

for model_name, history in histories_to_plot.items():
    if history.get('val_dice') and history.get('epoch') and history['val_dice']:
        val_epochs_dice = [e for e, vd in zip(history['epoch'], history['val_dice']) if not np.isnan(vd)]
        val_dices_data = [vd for vd in history['val_dice'] if not np.isnan(vd)]
        
        if val_epochs_dice:
            ax_val_dice.plot(val_epochs_dice, val_dices_data, linestyle='-', color=colors.get(model_name, 'black'), label=model_name, alpha=0.9, marker='.') # Προσθήκη marker
    else:
        print(f"Δεν βρέθηκαν δεδομένα validation dice για {model_name}")

ax_val_dice.set_xlabel("Epoch")
ax_val_dice.set_ylabel("Dice Score")
ax_val_dice.set_ylim(0, 1.0) # Όριο άξονα Υ από 0 έως 1
ax_val_dice.grid(True, linestyle=':')
ax_val_dice.legend()
ax_val_dice.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

# --- Manually Extracted Data from Logs (Epochs 1-8 and 11-20) ---
# Combined from the two log snippets provided
# Note: Epochs 9 & 10 from the second snippet seem to correspond to the FIRST run
# based on the loss values. Epochs 11-20 are from the SECOND run.
# We will combine 1-8 from the first run and 11-20 from the second run.

epochs =        [ 1,      2,      3,      4,      5,      6,      7,      8,      11,     12,     13,     14,     15,     16,     17,     18,     19,     20     ]
train_losses =  [ 2.1832, 1.5694, 1.4000, 1.3416, 1.2291, 1.1261, 1.0380, 1.0478, 0.4037, 0.3613, 0.3635, 0.3855, 0.3524, 0.3422, 0.3364, 0.3341, 0.3180, 0.3432 ]
val_losses =    [ 1.6963, 1.3675, 1.3393, 1.2625, 1.0884, 1.0441, 1.0637, 0.9770, 0.4060, 0.4051, 0.3919, 0.3254, 0.3404, 0.3276, 0.2946, 0.2834, 0.3322, 0.3220 ]
# Rotation accuracy is also available if you want to plot it
# val_rot_acc =   [ 0.1515, 0.2222, 0.3232, 0.2020, 0.2424, 0.3232, 0.2020, 0.3232, 0.2525, 0.2828, 0.2525, 0.2121, 0.1919, 0.2727, 0.2626, 0.2727, 0.2525, 0.2525 ]

# Define where to save the plot (optional)
output_dir = r"C:\Users\Kouts\Downloads\pretrain_logs.csv" 

# --- Plotting ---
plt.style.use('seaborn-v0_8-darkgrid')
plt.figure("Pretraining Loss Curves", figsize=(10, 5))

# Plot Training and Validation Loss
plt.plot(epochs, train_losses, marker='o', linestyle='-', color="red", label='Avg Train Loss')
plt.plot(epochs, val_losses, marker='s', linestyle='--', color="blue", label='Avg Validation Loss')

plt.title("Pretraining Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Total SSL Loss")
plt.grid(True, linestyle=':')
plt.legend()
plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True, min_n_ticks=10)) # Ensure integer ticks

plt.tight_layout()

# Save the plot (optional)
try:
    os.makedirs(output_dir, exist_ok=True) # Ensure directory exists
    plt.savefig(plot_save_path, dpi=300)
    print(f"Plot saved to: {plot_save_path}")
except Exception as e:
    print(f"Error saving plot: {e}")

plt.show()

# --- Optional: Save data to CSV ---
history_data = {
    "epoch": epochs,
    "train_loss": train_losses,
    "val_loss": val_losses,
    # "val_rot_acc": val_rot_acc # Uncomment if you want rotation accuracy too
}
history_csv_path = os.path.join(output_dir, "pretraining_history.csv")
try:
    df_history = pd.DataFrame(history_data)
    df_history.to_csv(history_csv_path, index=False)
    print(f"Pretraining history saved to: {history_csv_path}")
except Exception as e:
    print(f"Error saving pretraining history CSV: {e}")

In [ ]:
epochs =          [ 1,      2,      3,      4,      5,      6,      7,      8,      11,     12,     13,     14,     15,     16,     17,     18,     19,     20     ]
# Training Losses
train_total_loss= [ 2.1832, 1.5694, 1.4000, 1.3416, 1.2291, 1.1261, 1.0380, 1.0478, 0.4037, 0.3613, 0.3635, 0.3855, 0.3524, 0.3422, 0.3364, 0.3341, 0.3180, 0.3432 ]
train_mae_loss =  [ 0.2501, 0.1568, 0.1229, 0.1013, 0.0881, 0.0792, 0.0718, 0.0659, 0.0513, 0.0475, 0.0411, 0.0400, 0.0376, 0.0378, 0.0361, 0.0337, 0.0313, 0.0301 ]
train_rot_loss =  [ 2.7673, 1.7285, 1.4598, 1.4120, 1.4019, 1.4412, 1.4159, 1.4410, 1.4109, 1.3905, 1.3840, 1.4172, 1.4086, 1.3925, 1.3973, 1.3920, 1.3824, 1.4021 ] # Note: Lambda was 0 for epochs 11-20, so these didn't affect total loss then
train_con_loss =  [ 1.0888, 1.0941, 1.0922, 1.0667, 0.8782, 0.6506, 0.5146, 0.5213, 0.3517, 0.3133, 0.3220, 0.3450, 0.3143, 0.3039, 0.2998, 0.2999, 0.2863, 0.3127 ]
train_light_loss= [ 0.0515, 0.0137, 0.0107, 0.0097, 0.0094, 0.0095, 0.0092, 0.0080, 0.0057, 0.0054, 0.0048, 0.0049, 0.0048, 0.0048, 0.0047, 0.0045, 0.0047, 0.0046 ]

# Validation Losses (Only available for epochs 11-20 from logs)
val_total_loss =  [ 1.6963, 1.3675, 1.3393, 1.2625, 1.0884, 1.0441, 1.0637, 0.9770, 0.4060, 0.4051, 0.3919, 0.3254, 0.3404, 0.3276, 0.2946, 0.2834, 0.3322, 0.3220 ]
val_mae_loss =    [ np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.0478, 0.0432, 0.0396, 0.0414, 0.0391, 0.0387, 0.0346, 0.0310, 0.0294, 0.0309 ]
val_rot_loss =    [ np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 1.3542, 1.3637, 1.3829, 1.4234, 1.4165, 1.3685, 1.4066, 1.4176, 1.4083, 1.3894 ]
val_con_loss =    [ np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.3576, 0.3614, 0.3518, 0.2835, 0.3008, 0.2884, 0.2596, 0.2520, 0.3023, 0.2907 ]
val_light_loss =  [ np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0.0052, 0.0047, 0.0046, 0.0051, 0.0048, 0.0044, 0.0042, 0.0045, 0.0044, 0.0050 ]


# Define where to save the plot (optional)
output_dir = "/content/drive/MyDrive/Διπλωματικη/runs/ssl" # Directory where checkpoints were saved
plot_save_path = os.path.join(output_dir, "pretraining_individual_loss_curves.png")

# --- Plotting ---
plt.style.use('seaborn-v0_8-darkgrid')
fig, axs = plt.subplots(2, 2, figsize=(14, 10)) # Create 2x2 grid
fig.suptitle("Pretraining Individual Loss Components Over Epochs", fontsize=16)

# Plot MAE Loss (Train vs Val)
axs[0, 0].plot(epochs, train_mae_loss, marker='o', linestyle='-', color="blue", label='Train MAE Loss')
axs[0, 0].plot(epochs, val_mae_loss, marker='x', linestyle='--', color="cyan", label='Val MAE Loss')
axs[0, 0].set_title("Masked Autoencoder (MAE) Loss")
axs[0, 0].set_xlabel("Epoch")
axs[0, 0].set_ylabel("Loss")
axs[0, 0].grid(True, linestyle=':')
axs[0, 0].legend()
axs[0, 0].xaxis.set_major_locator(plt.MaxNLocator(integer=True, min_n_ticks=10))

# Plot Contrastive Loss (Train vs Val)
axs[0, 1].plot(epochs, train_con_loss, marker='o', linestyle='-', color="green", label='Train Contrastive Loss')
axs[0, 1].plot(epochs, val_con_loss, marker='x', linestyle='--', color="lightgreen", label='Val Contrastive Loss')
axs[0, 1].set_title("Contrastive (SimCLR) Loss")
axs[0, 1].set_xlabel("Epoch")
axs[0, 1].set_ylabel("Loss")
axs[0, 1].grid(True, linestyle=':')
axs[0, 1].legend()
axs[0, 1].xaxis.set_major_locator(plt.MaxNLocator(integer=True, min_n_ticks=10))

# Plot Light Reconstruction Loss (Train vs Val)
axs[1, 0].plot(epochs, train_light_loss, marker='o', linestyle='-', color="purple", label='Train Light Recon Loss')
axs[1, 0].plot(epochs, val_light_loss, marker='x', linestyle='--', color="violet", label='Val Light Recon Loss')
axs[1, 0].set_title("Lightweight Reconstruction Loss")
axs[1, 0].set_xlabel("Epoch")
axs[1, 0].set_ylabel("Loss")
axs[1, 0].grid(True, linestyle=':')
axs[1, 0].legend()
axs[1, 0].xaxis.set_major_locator(plt.MaxNLocator(integer=True, min_n_ticks=10))

# Plot Rotation Loss (Train vs Val)
axs[1, 1].plot(epochs, train_rot_loss, marker='o', linestyle='-', color="orange", label='Train Rotation Loss')
axs[1, 1].plot(epochs, val_rot_loss, marker='x', linestyle='--', color="yellow", label='Val Rotation Loss')
axs[1, 1].axhline(y=np.log(4), color='grey', linestyle=':', label='Chance Level (~1.386)') # Add chance level line
axs[1, 1].set_title("Rotation Prediction Loss")
axs[1, 1].set_xlabel("Epoch")
axs[1, 1].set_ylabel("Loss")
axs[1, 1].grid(True, linestyle=':')
axs[1, 1].legend()
axs[1, 1].xaxis.set_major_locator(plt.MaxNLocator(integer=True, min_n_ticks=10))

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout

# Save the plot (optional)
try:
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(plot_save_path, dpi=300)
    print(f"Plot saved to: {plot_save_path}")
except Exception as e:
    print(f"Error saving plot: {e}")

plt.show()

# --- Optional: Save data to CSV ---
history_data_individual = {
    "epoch": epochs,
    "train_total_loss": train_total_loss,
    "train_mae_loss": train_mae_loss,
    "train_rot_loss": train_rot_loss,
    "train_con_loss": train_con_loss,
    "train_light_loss": train_light_loss,
    "val_total_loss": val_total_loss,
    "val_mae_loss": val_mae_loss,
    "val_rot_loss": val_rot_loss,
    "val_con_loss": val_con_loss,
    "val_light_loss": val_light_loss,
    # "val_rot_acc": val_rot_acc # Uncomment if you want rotation accuracy too
}
history_csv_path_individual = os.path.join(output_dir, "pretraining_history_individual.csv")
try:
    df_history_individual = pd.DataFrame(history_data_individual)
    df_history_individual.to_csv(history_csv_path_individual, index=False)
    print(f"Pretraining history with individual losses saved to: {history_csv_path_individual}")
except Exception as e:
    print(f"Error saving pretraining history CSV: {e}")

In [ ]:
 Loading best model from: /content/drive/MyDrive/Διπλωματικη/runs/segmentation_finetune_custom/best_metric_model_custom.pt
Best model loaded successfully.

Visualizing Sample 1 (Slice 77)

Visualizing Sample 2 (Slice 77)

Visualizing Sample 3 (Slice 77)

Visualizing Sample 4 (Slice 77)

Visualizing Sample 5 (Slice 77)

Visualizing Sample 6 (Slice 77)

Visualizing Sample 7 (Slice 77)

Visualizing Sample 8 (Slice 77)

Visualizing Sample 9 (Slice 77)

Visualizing Sample 10 (Slice 77)

Visualizing Sample 11 (Slice 77)

Visualizing Sample 12 (Slice 77)

Visualizing Sample 13 (Slice 77)

Visualizing Sample 14 (Slice 77)

Visualizing Sample 15 (Slice 77)

Visualization complete.


In [ ]:
# %% Βήμα 2 (Διορθωμένο): Προετοιμασία Metadata και Δημιουργία Data Dicts

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from IPython.display import display
import os
import re
from glob import glob


metadata_path = r"/content/drive/MyDrive/Διπλωματικη/balanced_metadata_gradeaware.csv"  # Ενημέρωσε με τη σωστή διαδρομή
# --- 1. Φόρτωση του Ισορροπημένου Metadata ---
try:
    df_balanced = pd.read_csv(metadata_path)
    print(f"Φορτώθηκε το ισορροπημένο metadata. Σύνολο δειγμάτων: {len(df_balanced)}")
except FileNotFoundError:
    print(f"ΣΦΑΛΜΑ: Δεν βρέθηκε το αρχείο {metadata_path}.")
    # exit() 
    
# --- 2. Επιλογή Βασικών Στηλών ---
# Κρατάμε μόνο τις στήλες που θα χρησιμοποιήσουμε (ID, Targets, Features)
columns_to_keep = [
    'ID', 'Grade', 'Diagnosis', # IDs & Στόχοι
    'Age', 'Sex', 'MGMT_Status', 'MGMT_Index', # Features
    'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy'
]
# Φιλτράρουμε το DataFrame κρατώντας μόνο αυτές τις στήλες
df_processed = df_balanced[[col for col in columns_to_keep if col in df_balanced.columns]].copy()


# --- 3. Επεξεργασία Ειδικών Στηλών (π.χ. MGMT_Index) ---
print("\n--- Επεξεργασία 'MGMT_Index' ---")
if 'MGMT_Index' in df_processed.columns:
    # 3.1. Δημιουργία σημαίας (flag)
    df_processed['MGMT_Index_Is_Unknown'] = (df_processed['MGMT_Index'] == 'unknown').astype(int)
    # 3.2. Μετατροπή 'unknown' σε NaN
    df_processed['MGMT_Index'] = pd.to_numeric(df_processed['MGMT_Index'], errors='coerce')
    # 3.3. Συμπλήρωση (Imputation) με Median
    mgmt_index_median = df_processed['MGMT_Index'].median()
    df_processed['MGMT_Index'].fillna(mgmt_index_median, inplace=True)
    print(f"Η στήλη 'MGMT_Index' επεξεργάστηκε (Missing flag δημιουργήθηκε, κενά συμπληρώθηκαν με {mgmt_index_median:.2f}).")
else:
    print("Η στήλη 'MGMT_Index' δεν βρέθηκε.")

# --- 4. Διαχείριση Άλλων Ελλιπών Τιμών (NaN ή 'unknown') ---
print("\n--- Καθαρισμός Κατηγορικών Στηλών ---")
categorical_features_to_clean = ['Sex', 'MGMT_Status', 'Codeletion_1p19q', 'IDH', 'EOR', 'PreImaging_Biopsy']
for col in categorical_features_to_clean:
    if col in df_processed.columns:
        # Συμπλήρωση NaN (αν υπάρχουν)
        if df_processed[col].isnull().any():
            mode_val = df_processed[col].mode()[0]
            print(f"Συμπλήρωση NaN στη στήλη '{col}' με '{mode_val}'.")
            df_processed[col].fillna(mode_val, inplace=True)
        # Αντικατάσταση 'unknown' (αν υπάρχει)
        if (df_processed[col] == 'unknown').any():
            mode_val = df_processed[col].mode()[0]
            print(f"Αντικατάσταση 'unknown' στη στήλη '{col}' με '{mode_val}'.")
            df_processed[col].replace('unknown', mode_val, inplace=True)

# Συμπλήρωση NaN στην 'Age' (αν υπάρχει)
if 'Age' in df_processed.columns and df_processed['Age'].isnull().any():
    age_median = df_processed['Age'].median()
    print(f"Συμπλήρωση NaN στην 'Age' με {age_median}.")
    df_processed['Age'].fillna(age_median, inplace=True)

# --- 5. Κωδικοποίηση Κατηγορικών Χαρακτηριστικών (One-Hot Encoding) ---
categorical_features_to_encode = ['Sex', 'Codeletion_1p19q', 'IDH', 'MGMT_Status', 'EOR', 'PreImaging_Biopsy']
existing_categorical_features = [col for col in categorical_features_to_encode if col in df_processed.columns]
if existing_categorical_features:
    print(f"Εφαρμογή One-Hot Encoding στις: {existing_categorical_features}")
    df_processed = pd.get_dummies(df_processed, columns=existing_categorical_features, dummy_na=False)
else:
    print("Δεν βρέθηκαν κατηγορικές στήλες για one-hot encoding.")

# --- 6. Κανονικοποίηση Συνεχών Χαρακτηριστικών (Scaling) ---
scaler = StandardScaler()
continuous_features_to_scale = ['Age', 'MGMT_Index']
existing_continuous_features = [col for col in continuous_features_to_scale if col in df_processed.columns]
if existing_continuous_features:
    print(f"Κανονικοποίηση (Standard Scaling) στις: {existing_continuous_features}")
    df_processed[existing_continuous_features] = scaler.fit_transform(df_processed[existing_continuous_features])
else:
    print("Δεν βρέθηκαν συνεχείς στήλες για scaling.")

# --- 7. Κωδικοποίηση Ετικετών (Label Encoding) ---
print("\n--- Κωδικοποίηση Ετικετών (Targets) ---")
grade_encoder = LabelEncoder()
df_processed['Grade_Label'] = grade_encoder.fit_transform(df_processed['Grade'])
GRADE_CLASSES = grade_encoder.classes_
NUM_CLASSES_GRADE = len(GRADE_CLASSES)
print(f"Κλάσεις Βαθμού (Grade): {list(GRADE_CLASSES)} -> {list(range(NUM_CLASSES_GRADE))}")

diag_encoder = LabelEncoder()
df_processed['Diagnosis_Label'] = diag_encoder.fit_transform(df_processed['Diagnosis'])
DIAG_CLASSES = diag_encoder.classes_
NUM_CLASSES_DIAG = len(DIAG_CLASSES)
print(f"Κλάσεις Διάγνωσης (Diagnosis): {list(DIAG_CLASSES)} -> {list(range(NUM_CLASSES_DIAG))}")

# --- 8. Ορισμός Τελικής Λίστας Χαρακτηριστικών ---
print("\n--- Ορισμός Τελικής Λίστας Χαρακτηριστικών ---")
# Ορίζουμε τις στήλες που ΔΕΝ είναι features
non_feature_cols = [
    'ID', 'Grade', 'Diagnosis', 'Grade_Label', 'Diagnosis_Label', 
    'OS', 'Survival_Status', 'BraTS21_ID', 'Segmentation_Cohort', 
    'MGMT_Cohort', 'MGMT_Label' # Αφαιρούμε όλες τις "επικίνδυνες" στήλες
]
# Οι τελικές στήλες-features είναι όλες οι άλλες
clinical_feature_columns = [col for col in df_processed.columns if col not in non_feature_cols]

CLINICAL_FEATURE_SIZE = len(clinical_feature_columns)
print(f"\n--- Έτοιμο για το Μοντέλο ---")
print(f"CLINICAL_FEATURE_SIZE = {CLINICAL_FEATURE_SIZE}")
print(f"NUM_CLASSES_GRADE = {NUM_CLASSES_GRADE}")
print(f"NUM_CLASSES_DIAG = {NUM_CLASSES_DIAG}")

print(f"\n--- Τελικές Στήλες Χαρακτηριστικών ({CLINICAL_FEATURE_SIZE}): ---")
print(clinical_feature_columns)

print("\n--- Επεξεργασμένο DataFrame (5 πρώτες σειρές) ---")
display(df_processed.head())

# --- 9. Εύρεση Αρχείων και Δημιουργία Data Dicts (Από τους φακέλους Subset) ---
image_files_all = sorted(glob(os.path.join(image_dir_ft, "*.nii.gz")))
label_files_all = sorted(glob(os.path.join(label_dir_ft, "*_tumor_segmentation.nii.gz")))

image_path_map = {os.path.basename(f).split('_')[0]: f for f in image_files_all}
label_path_map = {os.path.basename(f).replace("_tumor_segmentation.nii.gz", ""): f for f in label_files_all}
balanced_ids = set(df_processed['ID'].astype(str))
data_dicts_mtl_balanced = []

for pid_csv in balanced_ids:
    match = re.match(r"([A-Za-z-]+-)(\d+)", pid_csv)
    if match:
        prefix, number_part = match.group(1), match.group(2)
        pid_formatted = f"{prefix}{number_part.zfill(4)}"
    else:
        pid_formatted = pid_csv
    
    if pid_formatted in image_path_map and pid_formatted in label_path_map:
        patient_meta = df_processed[df_processed['ID'] == pid_csv].iloc[0]
        
        data_dicts_mtl_balanced.append({
            image_key: image_path_map[pid_formatted],
            label_key: label_path_map[pid_formatted],
            clinical_key: torch.tensor(patient_meta[clinical_feature_columns].values.astype(np.float32), dtype=torch.float32),
            grade_key: torch.tensor(patient_meta['Grade_Label'], dtype=torch.long),
            diag_key: torch.tensor(patient_meta['Diagnosis_Label'], dtype=torch.long)
        })
    else:
        print(f"Warning: Δεν βρέθηκαν αρχεία για το ID {pid_csv} (formatted: {pid_formatted}) στους φακέλους subset.")

print(f"Δημιουργήθηκαν {len(data_dicts_mtl_balanced)} data dicts.")

# --- 10. Νέο Split (Train/Val/Test) ---
train_files_mtl, rest_files_mtl = partition_dataset(data_dicts_mtl_balanced, ratios=[0.7, 0.3], shuffle=True, seed=42)
val_files_mtl, test_files_mtl = partition_dataset(rest_files_mtl, ratios=[0.5, 0.5], shuffle=False, seed=42)
print(f"Νέο Split (MTL): {len(train_files_mtl)} train, {len(val_files_mtl)} val, {len(test_files_mtl)} test.")

In [ ]:
df_processed.info()

In [ ]:
df_processed.to_csv("processed_balanced_metadata_gradeaware.csv", index=False)
print("Το επεξεργασμένο metadata αποθηκεύτηκε επιτυχώς.")

In [ ]:
print(clinical_feature_columns)

In [ ]:


import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    ConfusionMatrixDisplay
)
print("--- Εκκίνηση Ablation Study: MLP Μόνο με Κλινικά Δεδομένα ---")

# --- 1. Ορισμός X (Features) και Y (Targets) ---
try:
    # X = Μόνο τα επεξεργασμένα κλινικά χαρακτηριστικά
    X = df_processed[clinical_feature_columns]
    
    # Y = Οι ετικέτες (labels) που δημιουργήσαμε
    Y_grade = df_processed['Grade_Label']
    Y_diag = df_processed['Diagnosis_Label']
    
    print(f"Βρέθηκαν {X.shape[0]} δείγματα με {X.shape[1]} κλινικά χαρακτηριστικά.")
    
except NameError:
    print("\n!!! ΣΦΑΛΜΑ: Οι μεταβλητές 'df_processed' ή 'clinical_feature_columns' δεν βρέθηκαν.")
    print("Παρακαλώ εκτελέστε πρώτα το προηγούμενο κελί 'Βήμα 2: Προετοιμασία Metadata'.")
    # (Σε κανονική εκτέλεση, θα βάζαμε exit() εδώ)
except Exception as e:
    print(f"Προέκυψε σφάλμα κατά την προετοιμασία των X/Y: {e}")

# --- 2. Διαχωρισμός Δεδομένων (Train/Test Split) ---
# (Χρησιμοποιούμε 30% για test set, με stratification για να διατηρηθεί η ισορροπία)

# Split για το Grade
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X, Y_grade, 
    test_size=0.3, 
    random_state=42, 
    stratify=Y_grade
)

# Split για το Diagnosis
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X, Y_diag, 
    test_size=0.3, 
    random_state=42, 
    stratify=Y_diag
)

## --- 3. Ορισμός "Πλέγματος" (Grid) Παραμέτρων για Αναζήτηση ---
# Βασιζόμαστε στην παρατήρησή σου για τα 3 layers
param_grid = {
    'hidden_layer_sizes': [
        (50,50),
        (100,50),
        (150,100),
        (200,100),
        (200,150),
        (200,200),
        (100, 50, 25),   # Μικρό 3-layer
        (150, 100, 50),  # Μεσαίο 3-layer
        (200, 100, 50),
        (200, 150, 50),
        (200, 200, 50),
        (200, 200, 100),
        (200, 200, 150),
        (200, 200, 200),
        (300, 300, 300)# Μεγάλο 3-layer# Μεγάλο 3-layer
    ],
    'alpha': [0.0001, 0.001, 0.01],  # L2 Regularization (για overfitting)
    'learning_rate_init': [0.001, 0.0001] # Αρχικός ρυθμός εκμάθησης
}

# --- Βασικό Μοντέλο MLP (με τις σταθερές παραμέτρους) ---
# Ορίζουμε τις παραμέτρους που ΔΕΝ θα αλλάξουν στο Grid Search
base_mlp = MLPClassifier(
    max_iter=1000, 
    random_state=42,
    activation='relu',
    solver='adam',
    early_stopping=True,  # Σημαντικό για ταχύτητα
    validation_fraction=0.1,
    n_iter_no_change=100 # Υπομονή 20 εποχών
)

print(f"Grid Search: Έλεγχος {len(param_grid['hidden_layer_sizes']) * len(param_grid['alpha']) * len(param_grid['learning_rate_init'])} συνδυασμών παραμέτρων...")

# --- 4. Grid Search για το 'Grade' ---
print("\nΕκκίνηση Grid Search για 'Grade'...")
# Χρησιμοποιούμε 3-fold Cross-Validation (cv=3)
# 'f1_macro' είναι καλή μετρική για μη-ισορροπημένα δεδομένα (όπως το Grade 3)
grid_search_grade = GridSearchCV(
    estimator=base_mlp,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro', # Εστίαση στο F1-score λόγω ανισορροπίας
    n_jobs=-1, # Χρήση όλων των πυρήνων CPU
    verbose=1  # Δείξε πρόοδο
)
grid_search_grade.fit(X_train_g, y_train_g)

print("\n--- Αποτελέσματα Grid Search (Grade) ---")
print(f"Καλύτερες Παράμετροι: {grid_search_grade.best_params_}")
print(f"Καλύτερο F1-Score (στο training set): {grid_search_grade.best_score_:.4f}")


# --- 5. Grid Search για το 'Diagnosis' ---
print("\nΕκκίνηση Grid Search για 'Diagnosis'...")
grid_search_diag = GridSearchCV(
    estimator=base_mlp,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro', # Χρησιμοποιούμε F1 και εδώ
    n_jobs=-1,
    verbose=1
)
grid_search_diag.fit(X_train_d, y_train_d)

print("\n--- Αποτελέσματα Grid Search (Diagnosis) ---")
print(f"Καλύτερες Παράμετροι: {grid_search_diag.best_params_}")
print(f"Καλύτερο F1-Score (στο training set): {grid_search_diag.best_score_:.4f}")


# --- 6. Τελική Αξιολόγηση στα Test Sets ---

# Παίρνουμε τα *καλύτερα* μοντέλα που βρήκε το Grid Search
best_mlp_grade = grid_search_grade.best_estimator_
best_mlp_diag = grid_search_diag.best_estimator_

print(f"\nΤο MLP 'Grade' (βέλτιστο) έτρεξε για {best_mlp_grade.n_iter_} εποχές.")
print(f"Το MLP 'Diagnosis' (βέλτιστο) έτρεξε για {best_mlp_diag.n_iter_} εποχές.")

# --- Αξιολόγηση 'Grade' ---
print("\n" + "="*40)
print(" ΑΞΙΟΛΟΓΗΣΗ 'GRADE' (Βέλτιστο Μοντέλο) ")
print("="*40)
y_pred_g = best_mlp_grade.predict(X_test_g)

print(f"\nΣυνολική Ακρίβεια (Accuracy) στο Test Set: {accuracy_score(y_test_g, y_pred_g):.4f}")
print("\nClassification Report (Grade) στο Test Set:")
print(classification_report(y_test_g, y_pred_g, target_names=GRADE_CLASSES.astype(str)))

print("Confusion Matrix (Grade) στο Test Set:")
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_estimator(
    best_mlp_grade, X_test_g, y_test_g, 
    display_labels=GRADE_CLASSES.astype(str), 
    cmap='Blues', 
    ax=ax
)
plt.title("Confusion Matrix - Grade (MLP Βέλτιστο)")
plt.show()

# --- Αξιολόγηση 'Diagnosis' ---
print("\n" + "="*40)
print(" ΑΞΙΟΛΟΓΗΣΗ 'DIAGNOSIS' (Βέλτιστο Μοντέλο) ")
print("="*40)
y_pred_d = best_mlp_diag.predict(X_test_d)

print(f"\nΣυνολική Ακρίβεια (Accuracy) στο Test Set: {accuracy_score(y_test_d, y_pred_d):.4f}")
print("\nClassification Report (Diagnosis) στο Test Set:")
print(classification_report(y_test_d, y_pred_d, target_names=DIAG_CLASSES.astype(str)))

print("Confusion Matrix (Diagnosis) στο Test Set:")
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_estimator(
    best_mlp_diag, X_test_d, y_test_d, 
    display_labels=DIAG_CLASSES.astype(str), 
    cmap='Blues', 
    ax=ax,
    xticks_rotation='vertical'
)
plt.title("Confusion Matrix - Diagnosis (MLP Βέλτιστο)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    ConfusionMatrixDisplay
)
from IPython.display import display
import shap


print("--- Εκκίνηση Ablation Study (Cross-Feature): MLP με Grade <-> Diagnosis ---")

# --- 1. Ορισμός X (Features) και Y (Targets) ---
try:
    # Βεβαιώσου ότι το df_processed και το clinical_feature_columns υπάρχουν από το προηγούμενο κελί
    _ = df_processed
    _ = clinical_feature_columns
except NameError:
    print("\n!!! ΣΦΑΛΜΑ: Οι μεταβλητές 'df_processed' ή 'clinical_feature_columns' δεν βρέθηκαν.")
    print("Παρακαλώ εκτελέστε πρώτα το κελί 'Βήμα 2: Προετοιμασία Metadata'.")
    # exit() # Θα σταματούσε εδώ

# --- 2. Δημιουργία Νέων One-Hot Features (Targets as Features) ---

# Κάνε one-hot encoding το 'Grade' (το original, όχι το _Label)
df_feat_grade = pd.get_dummies(df_processed['Grade'], prefix='Feat_Grade')
grade_feature_names = df_feat_grade.columns.tolist()
print(f"\nΔημιουργήθηκαν {len(grade_feature_names)} features από το 'Grade': {grade_feature_names}")

# Κάνε one-hot encoding το 'Diagnosis' (το original)
df_feat_diag = pd.get_dummies(df_processed['Diagnosis'], prefix='Feat_Diag')
diag_feature_names = df_feat_diag.columns.tolist()
print(f"Δημιουργήθηκαν {len(diag_feature_names)} features από το 'Diagnosis'.")

# Συνένωση των νέων features με το df_processed
df_processed_xfeat = pd.concat([df_processed, df_feat_grade, df_feat_diag], axis=1)

# --- 3. Ορισμός Νέων Συνόλων Χαρακτηριστικών (Feature Sets) ---

# Features για την πρόβλεψη 'Grade' = Βασικά Κλινικά + Feat_Diag
X_features_for_grade = clinical_feature_columns + diag_feature_names
X_g = df_processed_xfeat[X_features_for_grade]
Y_g = df_processed_xfeat['Grade_Label']
print(f"\nTask 'Grade': {X_g.shape[1]} features (Βασικά + Διάγνωση).")

# Features για την πρόβλεψη 'Diagnosis' = Βασικά Κλινικά + Feat_Grade
X_features_for_diag = clinical_feature_columns + grade_feature_names
X_d = df_processed_xfeat[X_features_for_diag]
Y_d = df_processed_xfeat['Diagnosis_Label']
print(f"Task 'Diagnosis': {X_d.shape[1]} features (Βασικά + Βαθμός).")


# --- 4. Διαχωρισμός Δεδομένων (Train/Test Split) ---

# Split για το Grade (με τα νέα X_g)
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_g, Y_g, 
    test_size=0.3, 
    random_state=42, 
    stratify=Y_g
)

# Split για το Diagnosis (με τα νέα X_d)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_d, Y_d, 
    test_size=0.3, 
    random_state=42, 
    stratify=Y_d
)

# --- 5. Grid Search (Όπως πριν) ---

# Ορισμός "Πλέγματος" (Grid)
param_grid = {
    'hidden_layer_sizes': [
        (100, 50, 25),   # Μικρό 3-layer
        (150, 100, 50),  # Μεσαίο 3-layer
        (200, 100, 50),
        (200, 150, 50),
        (200, 200, 50),
        (200, 200, 100),
        (200, 200, 150),
        (200, 200, 200),
        (300, 300, 300)# Μεγάλο 3-layer# Μεγάλο 3-layer
    ],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.0001]
}

# Βασικό Μοντέλο MLP
base_mlp = MLPClassifier(
    max_iter=1000, 
    random_state=42,
    activation='relu',
    solver='adam',
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=40
)

# --- Grid Search για το 'Grade' (με features Διάγνωσης) ---
print("\nΕκκίνηση Grid Search για 'Grade' (με 'Diagnosis' ως feature)...")
grid_search_grade = GridSearchCV(
    estimator=base_mlp, param_grid=param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search_grade.fit(X_train_g, y_train_g)
print(f"Καλύτερες Παράμετροι (Grade): {grid_search_grade.best_params_}")
print(f"Καλύτερο F1-Score (στο training set): {grid_search_grade.best_score_:.4f}")

# --- Grid Search για το 'Diagnosis' (με features Βαθμού) ---
print("\nΕκκίνηση Grid Search για 'Diagnosis' (με 'Grade' ως feature)...")
grid_search_diag = GridSearchCV(
    estimator=base_mlp, param_grid=param_grid, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1
)
grid_search_diag.fit(X_train_d, y_train_d)
print(f"Καλύτερες Παράμετροι (Diagnosis): {grid_search_diag.best_params_}")
print(f"Καλύτερο F1-Score (στο training set): {grid_search_diag.best_score_:.4f}")

# --- 6. Τελική Αξιολόγηση στα Test Sets ---

best_mlp_grade = grid_search_grade.best_estimator_
best_mlp_diag = grid_search_diag.best_estimator_

print(f"\nΤο MLP 'Grade' (βέλτιστο, με 'Diagnosis') έτρεξε για {best_mlp_grade.n_iter_} εποχές.")
print(f"Το MLP 'Diagnosis' (βέλτιστο, με 'Grade') έτρεξε για {best_mlp_diag.n_iter_} εποχές.")

# --- Αξιολόγηση 'Grade' ---
print("\n" + "="*40)
print(" ΑΞΙΟΛΟΓΗΣΗ 'GRADE' (με 'Diagnosis' ως feature) ")
print("="*40)
y_pred_g = best_mlp_grade.predict(X_test_g)
print(f"\nΣυνολική Ακρίβεια (Accuracy) στο Test Set: {accuracy_score(y_test_g, y_pred_g):.4f}")
print("\nClassification Report (Grade) στο Test Set:")
print(classification_report(y_test_g, y_pred_g, target_names=GRADE_CLASSES.astype(str)))
print("Confusion Matrix (Grade) στο Test Set:")
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_estimator(
    best_mlp_grade, X_test_g, y_test_g, 
    display_labels=GRADE_CLASSES.astype(str), cmap='Blues', ax=ax
)
plt.title("Confusion Matrix - Grade (MLP με 'Diagnosis' feature)")
plt.show()

# --- Αξιολόγηση 'Diagnosis' ---
print("\n" + "="*40)
print(" ΑΞΙΟΛΟΓΗΣΗ 'DIAGNOSIS' (με 'Grade' ως feature) ")
print("="*40)
y_pred_d = best_mlp_diag.predict(X_test_d)
print(f"\nΣυνολική Ακρίβεια (Accuracy) στο Test Set: {accuracy_score(y_test_d, y_pred_d):.4f}")
print("\nClassification Report (Diagnosis) στο Test Set:")
print(classification_report(y_test_d, y_pred_d, target_names=DIAG_CLASSES.astype(str)))
print("Confusion Matrix (Diagnosis) στο Test Set:")
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_estimator(
    best_mlp_diag, X_test_d, y_test_d, 
    display_labels=DIAG_CLASSES.astype(str), cmap='Blues', ax=ax, xticks_rotation='vertical'
)
plt.title("Confusion Matrix - Diagnosis (MLP με 'Grade' feature)")
plt.show()


# --- 7. ΝΕΟ: Ανάλυση SHAP (Explainability) ---
print("\n" + "="*40)
print(" ΑΝΑΛΥΣΗ SHAP (Explainability) ")
print("="*40)
print("Υπολογισμός SHAP values (Αυτό μπορεί να πάρει λίγο χρόνο)...")

# --- SHAP για 'Grade' ---
# 1. Δημιουργία "background" dataset (π.χ. 50 τυχαία δείγματα από το training set)
# Χρησιμοποιούμε DataFrames για να κρατήσουμε τα ονόματα των στηλών
X_train_g_df = pd.DataFrame(X_train_g, columns=X_features_for_grade)
X_test_g_df = pd.DataFrame(X_test_g, columns=X_features_for_grade)
background_g = shap.sample(X_train_g_df, 50) 

# 2. Δημιουργία του Explainer
explainer_grade = shap.KernelExplainer(best_mlp_grade.predict_proba, background_g)

# 3. Υπολογισμός SHAP values για το test set
shap_values_grade = explainer_grade.shap_values(X_test_g_df)

print("\n--- SHAP Summary Plot (Grade) ---")
# Σχεδίαση summary plot (bar plot) για τη συνολική σημασία των features
# Επειδή έχουμε 3 κλάσεις (Grades), μπορούμε να δούμε τη σημασία για κάθε κλάση
# Ή να πάρουμε τον μέσο όρο της απόλυτης τιμής
shap.summary_plot(shap_values_grade, X_test_g_df, plot_type="bar", class_names=GRADE_CLASSES.astype(str))
plt.title("Συνολική Σημασία Χαρακτηριστικών (SHAP) για 'Grade'")
plt.show()

# --- SHAP για 'Diagnosis' ---
# 1. Background dataset
X_train_d_df = pd.DataFrame(X_train_d, columns=X_features_for_diag)
X_test_d_df = pd.DataFrame(X_test_d, columns=X_features_for_diag)
background_d = shap.sample(X_train_d_df, 50)

# 2. Explainer
explainer_diag = shap.KernelExplainer(best_mlp_diag.predict_proba, background_d)

# 3. Υπολογισμός SHAP values
shap_values_diag = explainer_diag.shap_values(X_test_d_df)

print("\n--- SHAP Summary Plot (Diagnosis) ---")
# Σχεδίαση summary plot (bar plot)
shap.summary_plot(shap_values_diag, X_test_d_df, plot_type="bar", class_names=DIAG_CLASSES.astype(str))
plt.title("Συνολική Σημασία Χαρακτηριστικών (SHAP) για 'Diagnosis'")
plt.show()

print("\nΗ ανάλυση SHAP ολοκληρώθηκε.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Parameters
theta1_old = 2.0
theta2_old = 3.0

theta1_new = 3/7
theta2_new = 5.0

# Data points (x1, x2)
# Σημείωση: Το 3ο σημείο έχει άγνωστο x2 (z). Βάζουμε μια ενδεικτική τιμή (π.χ. 1.5) για την απεικόνιση.
data_points = np.array([
    [1, 2],
    [4, 5],
    [2, 1.5] 
])

# Grid setup
x1 = np.linspace(0, 6, 100)
x2 = np.linspace(0, 6, 100)
X1, X2 = np.meshgrid(x1, x2)

# --- PDF Functions ---
def joint_pdf(x1, x2, t1, t2):
    # P(x1) = t1 * exp(-t1 * x1)
    p_x1 = t1 * np.exp(-t1 * x1)
    # P(x2) = 1/t2 if 0 <= x2 <= t2 else 0
    p_x2 = np.where((x2 >= 0) & (x2 <= t2), 1/t2, 0)
    return p_x1 * p_x2

Z_old = joint_pdf(X1, X2, theta1_old, theta2_old)
Z_new = joint_pdf(X1, X2, theta1_new, theta2_new)

# --- Plotting ---
fig = plt.figure(figsize=(16, 8))

# Plot 1: Initial Model
ax1 = fig.add_subplot(121, projection='3d')
surf1 = ax1.plot_surface(X1, X2, Z_old, cmap='Reds', alpha=0.7, edgecolor='none')
ax1.set_title(f'Initial Model\n$\\theta_1={theta1_old}, \\theta_2={theta2_old}$')
ax1.set_xlabel('$x_1$ (Exponential)')
ax1.set_ylabel('$x_2$ (Uniform)')
ax1.set_zlabel('Probability Density')

# Add data points
# Στο παλιό μοντέλο, το (4,5) έχει πιθανότητα 0
ax1.scatter(data_points[:,0], data_points[:,1], [0, 0, 0], color='blue', s=50, label='Data Points', depthshade=False)
ax1.text(4, 5, 0.1, " Impossible (4,5)", color='black')

# Plot 2: New Model
ax2 = fig.add_subplot(122, projection='3d')
surf2 = ax2.plot_surface(X1, X2, Z_new, cmap='Greens', alpha=0.7, edgecolor='none')
ax2.set_title(f'New Model (After EM)\n$\\theta_1={theta1_new:.3f}, \\theta_2={theta2_new}$')
ax2.set_xlabel('$x_1$ (Exponential)')
ax2.set_ylabel('$x_2$ (Uniform)')
ax2.set_zlabel('Probability Density')

# Add data points on the surface
z_vals_new = joint_pdf(data_points[:,0], data_points[:,1], theta1_new, theta2_new)
ax2.scatter(data_points[:,0], data_points[:,1], z_vals_new, color='blue', s=50, label='Data Points', depthshade=False)

plt.tight_layout()
plt.show()

In [ ]:
pip install pyradiomics

In [ ]:
pip install simpleitk

In [ ]:
import os
import glob
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor
import numpy as np

# --- 1. CONFIGURATION ---
# Path to your dataset root folder
DATA_ROOT = "./dataset_directory" 
OUTPUT_CSV = "radiomics_features_raw.csv"

# Define filenames (Change these to match your exact file names)
IMG_NAME = "image.nii.gz"  # T1ce or FLAIR (whichever you use for grading)
MASK_NAME = "mask.nii.gz"  # The binary segmentation mask

# --- 2. RADIOMICS SETTINGS (Optimized for MRI) ---
# MRI intensities are arbitrary, so we MUST normalize them.
settings = {
    # Normalization: Scales intensities to fit a standard range (e.g., -3 to +3 sigma)
    # This ensures that "Texture" means the same thing across different scanners.
    'normalize': True,
    'normalizeScale': 100,  # Maps the range to roughly [-300, 300]
    
    # Resampling: Standardizes voxel size (critical for 3D shape features)
    'resampledPixelSpacing': [1, 1, 1], # Resample to 1x1x1 mm isotropic
    'interpolator': sitk.sitkBSpline,   # High-quality interpolation
    
    # Discretization: Groups grey values into bins for texture calculation
    # With normalizeScale=100, a binWidth of 5 yields ~120 bins (Ideal for MRI)
    'binWidth': 5,
    
    # Misc
    'voxelArrayShift': 300, # Shift values to avoid negative numbers (optional but safe)
    'verbose': True
}

def get_extractor():
    extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
    
    # Enable the "Grading" feature classes
    extractor.disableAllFeatures()
    extractor.enableFeatureClassByName('firstorder') # Entropy, Skewness (Intensity)
    extractor.enableFeatureClassByName('shape')      # Volume, Surface Area (Growth)
    extractor.enableFeatureClassByName('glcm')       # Contrast, Homogeneity (Texture)
    extractor.enableFeatureClassByName('glrlm')      # Run Length (Heterogeneity)
    extractor.enableFeatureClassByName('glszm')      # Zone Size (Clumping)
    extractor.enableFeatureClassByName('gldm')       # Dependence (Coarseness)
    
    return extractor

def process_dataset():
    extractor = get_extractor()
    data_list = []
    
    # Find all patient folders
    subjects = sorted(glob.glob(os.path.join(DATA_ROOT, "*")))
    print(f"Found {len(subjects)} subjects. Starting extraction...")

    for subj_path in subjects:
        if not os.path.isdir(subj_path): continue
        
        subj_id = os.path.basename(subj_path)
        image_path = os.path.join(subj_path, IMG_NAME)
        mask_path = os.path.join(subj_path, MASK_NAME)
        
        # Validation checks
        if not os.path.exists(image_path):
            print(f"⚠️ Skipping {subj_id}: Image not found.")
            continue
        if not os.path.exists(mask_path):
            print(f"⚠️ Skipping {subj_id}: Mask not found.")
            continue

        try:
            print(f"Processing {subj_id}...", end="\r")
            
            # Run Extraction
            result = extractor.execute(image_path, mask_path)
            
            # Clean Dictionary (Remove 'general_info_...' metadata keys)
            clean_row = {'patient_id': subj_id}
            for key, value in result.items():
                if "diagnostic" not in key: # Remove generic metadata
                    # Convert numpy types to python float for clean CSV
                    if isinstance(value, (np.ndarray, np.generic)):
                        value = value.item()
                    clean_row[key] = value
            
            data_list.append(clean_row)
            
        except Exception as e:
            print(f"\n❌ Error on {subj_id}: {e}")

    # Save to CSV
    if data_list:
        df = pd.DataFrame(data_list)
        # Reorder columns to put ID first
        cols = ['patient_id'] + [c for c in df.columns if c != 'patient_id']
        df = df[cols]
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"\n✅ Success! Saved {len(df)} patients to {OUTPUT_CSV}")
    else:
        print("\n❌ No features extracted. Check your paths.")

if __name__ == "__main__":
    process_dataset()